In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# 필터링 실습

이 노트북에서는 시간 영역 신호에서 원하는 주파수 대역만 추출하거나, 불필요한 저주파 드리프트와 고주파 잡음을 제거하는 필터링을 실습한다.

필터는 `utils.filtering_zerophase(x, fs, ftype, ...)` 한 함수로 적용한다. 이 함수는 `scipy.signal.butter` 로 설계한 Butterworth 필터를 `sosfiltfilt` 로 앞뒤 두 번 통과시키는 영위상(zero-phase) 필터이다. 인과(causal) 필터인 `utils.filtering` 은 실습 2의 위상 지연 비교에만 쓴다.

**Part A. 기본 개념**: 3 Hz, 10 Hz, 15 Hz 의 세 사인파를 더한, 잡음이 없는 합성 신호를 쓴다.
- 실습 1. 주파수 분석으로 신호 구성 확인하기
- 실습 2. BPF 로 한 대역만 통과시키기, 인과 필터와 영위상 필터의 비교
- 실습 3. HPF 로 저주파 사인파 제거
- 실습 4. LPF 로 고주파 사인파 제거
- 실습 5. HPF + LPF ≈ 원 신호 검증, 상보성 관찰

**Part B. 응용 시나리오**: 드리프트, 공진, 잡음이 섞인 신호를 쓴다.
- 실습 6. 신호 구성: 드리프트 + 공진 + 잡음
- 실습 7. HPF 로 드리프트 제거
- 실습 8. BPF 로 공진 대역 추출
- 실습 9. 신호 분해: 드리프트, 공진, 잡음의 세 성분 분리

## 준비: 라이브러리와 데이터 불러오기

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils import fft, filtering, filtering_zerophase

In [ ]:
fs = 1000  # sampling frequency (Hz)

data = np.array(pd.read_csv('./data/data_sample_filtering.csv'))
print('shape:', data.shape)

t = data[:, 1]
v = data[:, 2]

---

## 실습 1. 주파수 분석으로 신호 구성 확인하기

필터를 설계하기 전에 **어떤 주파수 성분이 얼마나 섞여 있는가**를 먼저 파악한다.
`utils.fft(v, fs)` 는 단측 진폭 스펙트럼 `(f, A)` 를 반환한다.

위 패널은 시간 영역이고, 아래 패널은 0 ~ 30 Hz 를 확대한 주파수 영역이다. 시간 파형만으로는 몇 개의 성분이 섞였는지 세기 어렵지만, 스펙트럼에서는 성분의 개수와 주파수를 바로 읽을 수 있다.

In [ ]:
f, A = fft(v, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v, 'C0')
ax[0].set_xlabel('Time (s)')
ax[0].set_ylabel('y')
ax[0].set_title('Raw signal (time domain)')

ax[1].plot(f, A, 'C0')
ax[1].set_xlabel('Frequency (Hz)')
ax[1].set_ylabel('|Y|')
ax[1].set_xlim([0, 30])
ax[1].set_ylim([0, 1.2])
ax[1].set_title('Single-sided amplitude spectrum (0 to 30 Hz)')

fig.tight_layout()
plt.show()

**관찰.** 스펙트럼에서는 3 Hz, 10 Hz, 15 Hz 의 세 주파수에 진폭 1의 성분이 나타나고, 나머지 주파수에서는 진폭이 0이다. 즉 이 신호는 서로 다른 주파수의 사인파 세 개로 이루어져 있으며 잡음은 섞여 있지 않다. 차단 주파수를 5 Hz 로 정하면 세 성분은 5 Hz 를 기준으로 저주파 쪽의 1개(3 Hz)와 고주파 쪽의 2개(10 Hz, 15 Hz)로 나뉜다. 이후의 실습에서는 이 경계를 이용하여 원하는 성분만 골라내는 방법을 확인한다.

---

## 실습 2. BPF 로 한 대역만 통과시키기

`utils` 에는 필터를 적용하는 함수가 두 가지 있다.

- `filtering(x, fs, ftype, ...)`: `sosfilt` 로 신호를 한 번만 통과시키는 **인과(causal)** 방식이다. 실시간 시스템에 적용할 수 있지만 **위상 지연**이 누적된다.
- `filtering_zerophase(x, fs, ftype, ...)`: `sosfiltfilt` 로 신호를 앞뒤로 두 번 필터링하여 위상 지연을 상쇄하는 **영위상(zero-phase)** 방식이다. 비인과 필터이므로 사후 분석에만 쓴다. 크기 응답이 $|H|^2$ 이 되어 전이 구간의 기울기가 더 급하다.

두 방식을 같은 대역통과 필터(band-pass filter, BPF)에 적용하여 **위상 차이**를 확인한다.

> **실무 팁**: 사후 분석에는 `filtering_zerophase` 를 기본으로 쓴다. 필터 차수가 높거나 차단 주파수가 Nyquist 주파수에 매우 가까워 `filtering_zerophase` 가 수치적으로 발산하면, 인과 필터인 `filtering` 으로 우회한다. 이 노트북의 나머지 실습은 모두 `filtering_zerophase` 를 사용한다.

In [ ]:
v_bp_causal = filtering(v,           fs, 'band', f_low=5, f_high=12)
v_bp_zero   = filtering_zerophase(v, fs, 'band', f_low=5, f_high=12)

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t, v,           color='C0', alpha=0.4, label='Original')
ax[0].plot(t, v_bp_causal, color='C1', ls='--', lw=1.2, label='Causal (sosfilt)')
ax[0].set_xlim([0, 2]); ax[0].set_ylabel('Amplitude'); ax[0].legend()
ax[0].set_title('Causal filter — real delay ~125 ms (more than one 10 Hz period)')

ax[1].plot(t, v,         color='C0', alpha=0.4, label='Original')
ax[1].plot(t, v_bp_zero, color='C2', ls='--', lw=1.2, label='Zero-phase (sosfiltfilt)')
ax[1].set_xlim([0, 2]); ax[1].set_xlabel('Time (s)'); ax[1].set_ylabel('Amplitude'); ax[1].legend()
ax[1].set_title('Zero-phase filter — no delay, peaks aligned with the 10 Hz component')

plt.tight_layout(); plt.show()

**관찰**:
- 위 패널의 인과 필터(`sosfilt`): 필터 출력인 파선은 처음 약 0.2초 동안 0에서 서서히 커지고, 0.5초 무렵에야 일정한 모양에 이른다. 이는 출력이 입력보다 늦게 나온다는 것을 보여 준다. 그 뒤 파선의 피크는 원 신호에 포함된 10 Hz 성분과 같은 시각에 있는 아래 패널 파선의 피크보다 약 25 ms 늦게 나타난다. 그러나 이 25 ms 는 실제 지연이 아니다. 이 BPF 는 10 Hz 성분을 약 125 ms 지연시킨다. 이 지연은 10 Hz 의 한 주기(100 ms)보다 길기 때문에, 피크끼리 비교하면 한 주기를 뺀 나머지(약 25 ms)만 보인다. 피크의 어긋남만으로 지연의 크기를 읽으면 안 된다.
- 아래 패널의 영위상 필터(`sosfiltfilt`): 파선의 피크가 원 신호에 포함된 10 Hz 성분의 피크와 **같은 시각**에 나타나며, 어긋남은 5 ms 이내이다. 원 신호에서 0.42초와 1.42초에 있는 큰 피크와 비교하면 이를 바로 확인할 수 있다. 필터를 앞뒤로 두 번 통과시키면 위상이 정확히 상쇄된다.
- 다만 **신호 양 끝 경계**에는 `sosfiltfilt` 의 패딩(padding) 처리 때문에 영위상 필터에서도 작은 왜곡이 남는다. 이 그림에서는 0.02초의 첫 피크가 다른 피크보다 조금 높다. 실무에서는 양 끝의 일정 구간을 잘라 내고 분석한다.
- 사후 분석에서는 "언제 일어났는가"가 중요하므로 영위상 필터를 기본으로 쓴다. 실시간 처리를 하거나, 차단 주파수가 Nyquist 주파수에 매우 가까워 `sosfiltfilt` 가 발산할 때에만 인과 필터로 우회한다.

---

## 실습 3. HPF 로 저주파 사인파 제거

고역통과 필터(high-pass filter, HPF)는 DC 와 드리프트를 포함하여 차단 주파수 아래의 저주파 성분을 제거하고, 그 위의 성분은 유지할 때 사용한다. 이 신호에서는 3 Hz 사인파가 제거 대상이다.

```python
utils.filtering_zerophase(x, fs, 'high', f_cut=...)
```

차단 주파수 5 Hz 로 필터링한 뒤 결과를 비교한다. 시간 파형은 앞 2초만 확대하여 그린다.

In [ ]:
cutoff = 5
v_hp = filtering_zerophase(v, fs, 'high', f_cut=cutoff)

f_raw, A_raw = fft(v, fs)
f_hp,  A_hp  = fft(v_hp, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v,    'C0', alpha=0.5, label='Original')
ax[0].plot(t, v_hp, 'C1',            label=f'Highpass (cutoff={cutoff} Hz)')
ax[0].set_xlim([0, 2])
ax[0].set_xlabel('Time (s)')
ax[0].set_ylabel('y')
ax[0].legend()

ax[1].plot(f_raw, A_raw, 'C0', alpha=0.5, label='Original')
ax[1].plot(f_hp,  A_hp,  'C1',            label='Filtered')
ax[1].set_xlabel('Frequency (Hz)')
ax[1].set_ylabel('|Y|')
ax[1].set_xlim([0, 30])
ax[1].set_ylim([0, 1.3])
ax[1].legend()

fig.tight_layout()
plt.show()

**관찰.** 스펙트럼에서 3 Hz 성분은 제거되고 10 Hz 와 15 Hz 의 두 성분은 그대로 유지된다. 시간 영역에서는 3 Hz 의 느린 변동이 사라지고 10 Hz 와 15 Hz 가 겹친 빠른 진동만 남는다. HPF 는 0 Hz 도 차단대역에 포함하므로 신호의 평균도 0으로 만든다. 다만 이 신호는 평균이 처음부터 0이므로 그림에서는 차이가 나타나지 않는다.

---

## 실습 4. LPF 로 고주파 사인파 제거

저역통과 필터(low-pass filter, LPF)는 차단 주파수 아래의 성분만 남기고 그 위의 성분은 제거한다. 이 신호에서는 10 Hz 와 15 Hz 의 사인파가 제거 대상이다.

```python
utils.filtering_zerophase(x, fs, 'low', f_cut=...)
```

실습 3과 같은 차단 주파수 5 Hz 로 필터링하여 상보적인 결과를 확인한다. 시간 파형은 앞 2초만 확대하여 그린다.

In [ ]:
cutoff = 5
v_lp = filtering_zerophase(v, fs, 'low', f_cut=cutoff)

f_raw, A_raw = fft(v, fs)
f_lp,  A_lp  = fft(v_lp, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v,    'C0', alpha=0.5, label='Original')
ax[0].plot(t, v_lp, 'C1',            label=f'Lowpass (cutoff={cutoff} Hz)')
ax[0].set_xlim([0, 2])
ax[0].set_xlabel('Time (s)')
ax[0].set_ylabel('y')
ax[0].legend()

ax[1].plot(f_raw, A_raw, 'C0', alpha=0.5, label='Original')
ax[1].plot(f_lp,  A_lp,  'C1',            label='Filtered')
ax[1].set_xlabel('Frequency (Hz)')
ax[1].set_ylabel('|Y|')
ax[1].set_xlim([0, 20])
ax[1].set_ylim([0, 1.3])
ax[1].legend()

fig.tight_layout()
plt.show()

**관찰.** 스펙트럼에서 10 Hz 와 15 Hz 의 성분은 제거되고 3 Hz 성분만 남는다. 시간 파형은 3 Hz 의 단일 사인파가 되어 매끄러워진다. 실습 3의 결과와 더하면 원 신호가 복원될 것으로 기대할 수 있으며, 이 관계는 다음 실습에서 확인한다.

---

## 실습 5. HPF + LPF ≈ 원 신호 검증

고역통과 필터와 저역통과 필터의 결과를 더하면 원 신호가 되는지 시간 영역에서 검증한다. 두 필터는 모두 영위상 필터(`filtering_zerophase`)로 적용한다.

In [ ]:
# HP + LP ≈ Original 확인 (zero-phase 필터)
v_hp = filtering_zerophase(v, fs, 'high', f_cut=5)
v_lp = filtering_zerophase(v, fs, 'low',  f_cut=5)

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True, gridspec_kw={'width_ratios': [2, 1]})
for a in ax:
    a.plot(t, v, color='C0', alpha=0.5, lw=1.5, label='Original')
    a.plot(t, v_hp + v_lp, color='C2', ls='--', lw=1.0, label='Recovered (v_hp + v_lp)')
    a.set_xlabel('Time (s)')
ax[0].set_xlim([0, 2]); ax[0].set_ylim([-3.4, 4.2]); ax[0].set_ylabel('Amplitude')
ax[0].legend(loc='upper center', ncol=2)
ax[0].set_title('First 2 s')
ax[1].set_xlim([9, 10])
ax[1].set_title('Last 1 s (boundary)')
fig.suptitle('Zero-phase HP + LP vs original')

plt.tight_layout(); plt.show()

**관찰**:
- 내부 구간에서는 복원한 신호(파선)가 원 신호(실선)와 **거의 완벽히 일치**한다. $|H_{HP}(f)|^2 + |H_{LP}(f)|^2 = 1$ 이 성립하기 때문이다.
- 왼쪽 패널의 앞 2초에서는 시작 경계까지 두 곡선이 완전히 겹친다. 오차는 신호의 **끝 쪽**에만 남는다. 오른쪽 패널에서는 약 9.8초 이후부터 차이가 눈에 띄며, 마지막 샘플에서 약 1에 이른다. 이 오차는 `sosfiltfilt` 의 패딩 처리 때문에 생긴다. HPF 와 LPF 의 결과에는 각각 양 끝에 경계 왜곡이 있지만, 두 결과를 더하면 시작 쪽의 왜곡은 서로 상쇄된다. 이 상쇄는 이 신호에서만 일어나는 것이 아니라 `sosfiltfilt` 의 계산 방식에서 비롯된다. 경계 오차가 양 끝에 똑같이 생긴다고 가정하지 말고, 실무에서는 양 끝을 적당히 잘라 내고 분석한다.
- 시간 영역에서 완벽한 상보성이 필요하면 `v_hp = v - v_lp` 와 같이 빼서 구하는 단순한 방식을 대안으로 쓸 수 있다.

---

## 실습 6. 신호 구성: 드리프트 + 공진 + 잡음

실습 6 ~ 9에서는 드리프트, 공진, 잡음을 더한 **실제 측정 신호를 본뜬 합성 신호**에 필터링 워크플로를 적용해 본다.

새 데이터 `data_filtering_application.csv` 를 불러와 변수 `v` 에 다시 할당하고, 가공하기 전 신호의 시간 파형과 스펙트럼을 본다. 이 데이터는 회전기계의 가속도 신호를 모방한 합성 신호이며, 저주파 드리프트, 6 ~ 10 Hz 의 광대역 공진, 가우시안 잡음으로 이루어진다. 드리프트는 0.5 Hz(진폭 1.5)와 0.3 Hz(진폭 0.5)의 두 사인파의 합이다.

In [ ]:
data = np.array(pd.read_csv('./data/data_filtering_application.csv'))
t = data[:, 1]
v = data[:, 2]
print('shape:', data.shape)

f, A = fft(v, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v, 'C0', linewidth=0.6)
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('y')
ax[0].set_title('Application signal: drift + resonance + noise')

ax[1].plot(f, A, 'C0')
ax[1].set_xlim([0, 30]); ax[1].set_ylim([0, 1.7])
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('|Y|')
ax[1].set_title('Single-sided amplitude spectrum (0 to 30 Hz)')

fig.tight_layout(); plt.show()

**관찰.** 스펙트럼에서는 세 성분을 확인할 수 있다. (1) 드리프트 성분은 0 ~ 2 Hz 에 있고, 진폭은 0.5 Hz 에서 약 1.5, 0.3 Hz 에서 약 0.5이다. (2) 공진 대역은 6 ~ 10 Hz 에 있다. 진폭이 가장 큰 빈은 8.4 Hz 의 약 0.43이며, 대역 안의 빈은 대부분 0.1 ~ 0.35이다. (3) 그 밖의 주파수에는 평평한 잡음 바닥이 나타나며, 빈당 진폭은 약 0.005 ~ 0.016이다. 드리프트 성분과 잡음 바닥의 진폭은 약 100 ~ 300배 차이가 나므로, 이 세로축에서는 잡음 바닥이 0에 가까운 선으로 보인다. 실습 7 ~ 9에서는 이 세 성분을 단계별로 정리하고 분해한다.

---

## 실습 7. HPF 로 드리프트 제거

가장 먼저 제거할 것은 저주파 드리프트이다. 차단 주파수가 3 Hz 인 HPF 를 적용한다.

In [ ]:
v_hp = filtering_zerophase(v, fs, 'high', f_cut=3)

f_raw, A_raw = fft(v, fs)
f_hp,  A_hp  = fft(v_hp, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v,    'C0', alpha=0.5, lw=0.6, label='Original')
ax[0].plot(t, v_hp, 'C1', lw=0.6, label='Highpass (cutoff=3 Hz)')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('y'); ax[0].legend()

ax[1].plot(f_raw, A_raw, 'C0', alpha=0.5, label='Original')
ax[1].plot(f_hp,  A_hp,  'C1',            label='Filtered')
ax[1].set_xlim([0, 30]); ax[1].set_ylim([0, 1.7])
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('|Y|'); ax[1].legend()

fig.tight_layout(); plt.show()

**관찰.** 시간 영역에서는 드리프트에 의한 큰 변동이 사라지고 빠른 진동과 잡음만 남는다. 스펙트럼에서는 0.3 Hz 와 0.5 Hz 의 드리프트 성분이 제거되고 6 ~ 10 Hz 의 공진 대역은 그대로 유지된다. 드리프트가 제거된 이 신호에서 다음 단계의 분석을 시작할 수 있다.

---

## 실습 8. BPF 로 공진 대역 추출

드리프트와 잡음을 동시에 제거하는 데에는 BPF 한 줄로 충분하다. 통과대역을 5 ~ 12 Hz 로 정하여 공진 대역만 추출한다.

In [ ]:
v_bp = filtering_zerophase(v, fs, 'band', f_low=5, f_high=12)

f_raw, A_raw = fft(v, fs)
f_bp,  A_bp  = fft(v_bp, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v,    'C0', alpha=0.5, lw=0.6, label='Original')
ax[0].plot(t, v_bp, 'C2', lw=0.6, label='Bandpass (5-12 Hz)')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('y'); ax[0].legend()

ax[1].plot(f_raw, A_raw, 'C0', alpha=0.5, label='Original')
ax[1].plot(f_bp,  A_bp,  'C2',            label='Filtered')
ax[1].set_xlim([0, 30]); ax[1].set_ylim([0, 1.7])
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('|Y|'); ax[1].legend()

fig.tight_layout(); plt.show()

**관찰.** 시간 영역에는 6 ~ 10 Hz 의 공진 진동만 남고, 스펙트럼에도 6 ~ 10 Hz 의 공진 대역만 나타난다. 드리프트와 통과대역 밖의 잡음이 함께 제거되었으며, BPF 한 줄이 HPF 와 LPF 두 단계의 효과를 동시에 낸다.

---

## 실습 9. 신호 분해: 세 성분의 분리

같은 원 신호에서 드리프트, 공진, 잡음의 세 성분을 모두 분리해 본다. LPF 와 BPF 를 적용하는 두 줄과 산술 연산 한 줄이면 분해가 끝난다.

In [ ]:
drift     = filtering_zerophase(v, fs, 'low',  f_cut=3)
resonance = filtering_zerophase(v, fs, 'band', f_low=5, f_high=12)
noise     = v - drift - resonance

fig, ax = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
ax[0].plot(t, v,         'C0', lw=0.6); ax[0].set_ylabel('y')
ax[0].set_title('(a) Raw signal (drift + resonance + noise)'); ax[0].set_ylim([-4, 4])

ax[1].plot(t, drift,     'C1', lw=1.0); ax[1].set_ylabel('y')
ax[1].set_title('(b) drift (LPF cutoff 3 Hz)'); ax[1].set_ylim([-4, 4])

ax[2].plot(t, resonance, 'C2', lw=0.6); ax[2].set_ylabel('y')
ax[2].set_title('(c) resonance (BPF 5-12 Hz)'); ax[2].set_ylim([-4, 4])

ax[3].plot(t, noise,     'C3', lw=0.6); ax[3].set_ylabel('y')
ax[3].set_xlabel('Time (s)')
ax[3].set_title('(d) residual noise (raw - drift - resonance)'); ax[3].set_ylim([-4, 4])

fig.tight_layout(); plt.show()

**관찰**:
- (b) 드리프트: 0.5 Hz(진폭 1.5)와 0.3 Hz(진폭 0.5)의 두 사인파의 합이며, 변동의 진폭은 1 ~ 2 사이이다.
- (c) 공진: 빠른 진동의 진폭이 천천히 변하는 포락선 구조를 보인다.
- (d) 잔여 잡음: 평균이 0 부근인 가우시안 분포이며, 시간적 구조가 없는 백색 잡음이다. 처음 약 0.2초와 마지막 약 0.1초의 양 끝에서 나타나는 변동은 LPF 와 BPF 의 경계 왜곡이다.

세 성분은 각각 센서 기준선의 변동, 하우징의 구조 공진, 측정 잡음이라는 물리적 원인에 대응한다.

---